### Read & Explore

In [0]:
from pyspark.sql.functions import trim, col, when

# Read prd_info from Bronze
df = spark.read.table("`databricks-medallion-lakehouse`.bronze.prd_info")

print("="*70)
print("SILVER: prd_info Transformation")
print("="*70)
print(f"\nBronze rows: {df.count():,}")

print("\nCheck for data quality issues:")
print(f"  NULL prd_cost: {df.filter(col('prd_cost').isNull()).count()}")
print(f"  NULL prd_end_dt: {df.filter(col('prd_end_dt').isNull()).count()}")

print("\nFirst 5 rows (BEFORE cleaning):")
df.show(5, truncate=False)

### Handle NULLs & Duplicates

In [0]:
# Step 1: Trim string columns
df_clean = df.select([
    trim(col(c)).alias(c) if df.schema[c].dataType.simpleString() == "string" else col(c)
    for c in df.columns
])

# Step 2: For products with same prd_key, keep LATEST (by prd_start_dt)
# This handles duplicate product versions
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("prd_key").orderBy(desc("prd_start_dt"))
df_clean = df_clean.withColumn("rn", row_number().over(window_spec))
df_clean = df_clean.filter(col("rn") == 1).drop("rn")

print(f"\nAfter deduplication: {df_clean.count():,} rows")
print(f"Removed {397 - df_clean.count()} duplicate versions")

print("\nDuplicate product check (should be 0):")
df_clean.groupBy("prd_key").count().filter(col("count") > 1).show()

### Rename Columns & Handle NULLs

In [0]:
# Step 3: Rename columns to professional names
df_clean = df_clean.select(
    col("prd_id").alias("product_id"),
    col("prd_key").alias("product_key"),
    col("prd_nm").alias("product_name"),
    col("prd_cost").alias("cost"),
    col("prd_line").alias("product_line"),
    col("prd_start_dt").alias("start_date"),
    col("prd_end_dt").alias("end_date")
)

# Step 4: Handle NULL costs
# Business rule: If cost is NULL, mark as 0 (or could mark as "TBD")
df_clean = df_clean.withColumn(
    "cost",
    when(col("cost").isNull(), 0).otherwise(col("cost"))
)

print(f"\nFinal schema:")
df_clean.printSchema()

print(f"\nFirst 5 rows (FINAL):")
df_clean.show(5, truncate=False)

print(f"\nNULL costs now: {df_clean.filter(col('cost') == 0).count()}")

### Write to Silver

In [0]:
# Step 5: Deduplicate by product_id (just in case)
df_clean = df_clean.dropDuplicates(["product_id"])

rows_final = df_clean.count()
print(f"\nFinal rows: {rows_final:,}")

# Step 6: Write to Silver
silver_table = "`databricks-medallion-lakehouse`.silver.prd_info"

spark.sql(f"DROP TABLE IF EXISTS {silver_table}")

df_clean.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(silver_table)

print(f"✅ Written to Silver: {silver_table}")